# EPSM GeoJSON Processing Pipeline Demo

This notebook demonstrates how to use the EPSM pipeline to:
1. Download building footprints from DTCC
2. Fetch  GeoJSON files

**Authors:** Aaron Qiyu Liu and Sanjay Somanath

In [1]:
# Import required modules
import sys
from pathlib import Path

# Add parent directory to path to import geojson_processor
sys.path.insert(0, str(Path.cwd().parent))

from geojson_processor.dtcc_service import DTCCService

Select the bounds for what area to simulate by drawing a box on a map

In [ ]:
# Interactive map with automatic bounds extraction
from ipyleaflet import Map, DrawControl, basemaps
from ipywidgets import Output
from IPython.display import display

# Create output widget to display bounds
output = Output()

# Create map centered on Sweden
map_widget = Map(basemap=basemaps.OpenStreetMap.Mapnik, center=(58.35, 11.93), zoom=13)

# Configure draw control for rectangles only
draw_control = DrawControl()
draw_control.rectangle = {"shapeOptions": {"color": "#0000FF", "fillOpacity": 0.3}}
draw_control.polyline = {}
draw_control.polygon = {}
draw_control.circle = {}
draw_control.circlemarker = {}

# Store bounds globally
drawn_bounds = {}

def handle_draw(target, action, geo_json):
    """Extract bounds when rectangle is drawn"""
    with output:
        output.clear_output()
        if action == 'created' and geo_json['geometry']['type'] == 'Polygon':
            coords = geo_json['geometry']['coordinates'][0]
            lats = [coord[1] for coord in coords]
            lons = [coord[0] for coord in coords]
            
            drawn_bounds['south'] = min(lats)
            drawn_bounds['north'] = max(lats)
            drawn_bounds['west'] = min(lons)
            drawn_bounds['east'] = max(lons)
            
            print("✅ Rectangle bounds captured (WGS84):")
            print(f"  South: {drawn_bounds['south']:.6f}")
            print(f"  North: {drawn_bounds['north']:.6f}")
            print(f"  West: {drawn_bounds['west']:.6f}")
            print(f"  East: {drawn_bounds['east']:.6f}")

draw_control.on_draw(handle_draw)
map_widget.add_control(draw_control)

display(map_widget, output)

Map(center=[58.35, 11.93], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_o…

Output()

In [3]:
# Convert WGS84 bounds to EPSG:3006 for DTCC
from pyproj import Transformer

if drawn_bounds:
    transformer = Transformer.from_crs("EPSG:4326", "EPSG:3006", always_xy=True)
    
    west_3006, south_3006 = transformer.transform(drawn_bounds['west'], drawn_bounds['south'])
    east_3006, north_3006 = transformer.transform(drawn_bounds['east'], drawn_bounds['north'])
    
    bounds = {
        'west': int(west_3006),
        'south': int(south_3006),
        'east': int(east_3006),
        'north': int(north_3006)
    }
    
    print("✅ Converted to EPSG:3006 for DTCC:")
    print(bounds)
else:
    print("❌ Draw a rectangle on the map first")

✅ Converted to EPSG:3006 for DTCC:
{'west': 318795, 'south': 6472656, 'east': 319763, 'north': 6473364}


## Download Building Data from DTCC (Requires DTCC Library)

If you have the DTCC library installed, you can download building footprints directly from DTCC's database for any area in Sweden.

In [4]:
# Example: Download building data from DTCC for a specific area in Uddevalla
# Note: This requires the DTCC library to be installed

# Create working directory
work_dir = Path('dtcc_output')
work_dir.mkdir(exist_ok=True)

# Initialize DTCC service
dtcc = DTCCService(str(work_dir))

# Download city data (building footprints and terrain)
try:
    geojson_path, terrain_path = dtcc.download_city_data(
        west=bounds['west'],
        south=bounds['south'],
        east=bounds['east'],
        north=bounds['north'],
        epsg=3006
    )
    print(f"✅ Downloaded GeoJSON: {geojson_path}")
    print(f"✅ Terrain STL: {terrain_path}")
except Exception as e:
    print(f"❌ DTCC download failed: {e}")
    print("Note: DTCC library may not be installed. See Option 2 for local GeoJSON processing.")

2026-01-28 09:06:49,557 [dtcc io2] [WARNING] Unable to find assimp, reading and writing .dae and .fbx files will not work
 To install assimp please see the instructions at https://www.assimp.org
2026-01-28 09:06:54,748 [root] [WARNING] <function default_view at 0x0000027B3B79CCC0> Method view already exists, replacing it.
2026-01-28 09:06:54,749 [root] [WARNING] <function default_view at 0x0000027B3B79CCC0> Method view already exists, replacing it.
2026-01-28 09:06:54,750 [root] [WARNING] <function default_view at 0x0000027B3B79CCC0> Method view already exists, replacing it.
2026-01-28 09:06:54,752 [root] [WARNING] <function default_view at 0x0000027B3B79CCC0> Method view already exists, replacing it.
2026-01-28 09:06:54,753 [root] [WARNING] <function default_view at 0x0000027B3B79CCC0> Method view already exists, replacing it.
2026-01-28 09:06:54,756 [root] [WARNING] <function default_view at 0x0000027B3B79CCC0> Method view already exists, replacing it.
2026-01-28 09:06:54,757 [geojso